# Semi- and fully multiplicative MMMs with terms

This is a runnable review companion to [PR #3178](https://github.com/pymc-labs/pymc-marketing/pull/3178) and [issue #3088](https://github.com/pymc-labs/pymc-marketing/issues/3088). Start with the shorter [semi-multiplicative terms example](semi_multiplicative_demo_terms.ipynb). Here, both models use the same inputs but **separate outcomes generated from their own known truth**. Their AVM scores are checks of each recipe, not a model-selection contest.

The question throughout is: **what does the experimental terms branch provide, and what must an analyst still define?** The code tests model composition, fitting, prediction, scenario evaluation, attribution, curves, optimization, and persistence. It does not establish causal effects from observed data.

### Navigation

| Section | Review question |
| --- | --- |
| [1. Imports and settings](#imports) | Which dependencies, units, and switches govern the run? |
| [2. Data](#data) | Are the synthetic inputs, truths, and holdout separate? |
| [3. Models](#models) | Can terms express both response equations? |
| [4. Fitting and diagnostics](#fitting) | Do the priors and posterior fit behave sensibly? |
| [5. AVM and forecast](#avm) | Do predictions track training and held-out sales? |
| [6. Contributions](#contributions) | What can be read from the graph, and what must be calculated? |
| [7. Curves and returns](#curves) | Which curves are model components versus sales responses? |
| [8. Optimization](#optimization) | Can the fitted response drive a constrained plan? |
| [9. Persistence and review takeaways](#review) | What can be saved, and what remains an API decision? |

**Labels used below:** **Terms API** names behavior supplied by this branch. **Notebook policy** names an explicit choice or calculation in these cells. In particular, a named term is a graph output; it is not automatically a sales contribution. Section 6 tests how to *take contributions* under several different definitions.

Run cells in order with a kernel importing this checkout. The branch's `pyproject.toml` defines supported packages. Additional notebook packages are `ipykernel`, `nbformat`, `matplotlib`, `zarr`, and `watermark`; use pandas 3 or newer on this experimental branch. A prepared local kernel is `.venv-terms-demo/bin/python`. For another environment: `python -m pip install -e . zarr ipykernel watermark "pandas>=3"`.

AVM means **actual versus model**. Four chains are configured below; reduce sampling only for a wiring check. All data are generated locally.


<a id="imports"></a>
## 1. Imports and settings

Import terms, PyMC, and plotting tools. Then declare the run settings before generating data.


Import the terms API alongside PyMC, labeled arrays, and plotting libraries.


In [ ]:
import sys
from math import factorial
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.xtensor as ptx
import xarray as xr
from IPython.display import display
from pymc_extras.prior import Prior

from pymc_marketing.experimental.terms import GAM, Data, Equation, Problem
from pymc_marketing.mmm import GeometricAdstock, LogisticSaturation
from pymc_marketing.terms import Parameter, Transform

Set the synthetic-data seed, forecast horizon, curve grid, sampler, and optional run switches. Four chains are used for the review run.


In [ ]:
config = {
    "seed": 3088,
    "training_weeks": 156,
    "forecast_weeks": 12,
    "l_max": 6,
    "price_reference": 10.0,
    "distribution_reference": 0.8,
    "target_scale": 1000.0,
    "curve_weeks": 8,
    "curve_points": 21,
}
sampler_config = dict(
    chains=4,
    cores=1,
    tune=1000,
    draws=1000,
    target_accept=0.95,
    nuts_sampler="pymc",
    progressbar=True,
)
RUN_OPTIMIZATION = True
RUN_SAVE_LOAD = False
display(pd.Series(config, name="Configuration").to_frame())

Record package versions and the active kernel. Declare channel scales and market references once; later scenarios reuse these constants.


In [ ]:
import importlib.metadata

packages = ["pymc-marketing", "pymc", "pymc-extras", "pytensor", "arviz", "xarray"]
display(pd.Series({p: importlib.metadata.version(p) for p in packages}, name="Version"))
print("Kernel:", sys.executable)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.18,
    }
)
rng = np.random.default_rng(config["seed"])
channels = ["tv", "search"]
channel_scale = xr.DataArray(
    [100.0, 80.0], dims="channel", coords={"channel": channels}
)
target_scale = config["target_scale"]
price_ref = config["price_reference"]
distribution_ref = config["distribution_reference"]

### Units and references

**Notebook policy:** spend is currency per week and sales are units per week. Divide spend by fixed channel scales and sales by a fixed target scale for fitting. Price and distribution stay in their original units. Keep the scales and reference values fixed across fitting, forecasts, and counterfactuals. The observed `y` and `expected_sales` remain in **thousands of sales** because the fitted target is scaled. The model recipes also use `.named()` to store `sales_units` for both models and `media_sales_units` / `baseline_sales_units` for the semi model. Terms do not assign units automatically: these expressions multiply by the declared `target_scale`. The full model’s per-channel sales-unit removal effects are defined later in the reporting graph, after an intervention policy is chosen. Alternatively, a model fitted to unscaled sales would need priors and observation noise in sales units. ROAS below means **sales units per currency unit**, not revenue return.


<a id="data"></a>
## 2. Data

Generate common spend, price, and distribution inputs. Give each model a separate synthetic sales series from its own known equation. Keep the last 12 weeks out of both fits.


Simulate 156 training weeks and 12 held-out weeks of spend, price, and distribution. Check that the two ratio inputs remain positive.


In [ ]:
n_train = config["training_weeks"]
n_total = n_train + config["forecast_weeks"]
dates = pd.date_range("2022-01-03", periods=n_total, freq="W-MON")
spend_scaled = rng.uniform(0.05, 1.0, size=(n_total, len(channels)))
spend_scaled *= rng.random(spend_scaled.shape) > 0.18
price = price_ref * rng.uniform(0.8, 1.2, n_total)
distribution = rng.uniform(0.60, 0.98, n_total)
inputs = xr.Dataset(
    {
        "spend": (("date", "channel"), spend_scaled),
        "price": ("date", price),
        "distribution": ("date", distribution),
    },
    coords={"date": dates, "channel": channels},
)
np.testing.assert_array_less(0, inputs["price"])
np.testing.assert_array_less(0, inputs["distribution"])

Declare the known baseline, channel response, market elasticity, and noise settings. These values generate the data and support later checks.


In [ ]:
truth_config = {
    "baseline": 1000.0,
    "noise_sigma": 50.0,
    "alpha": np.array([0.35, 0.15]),
    "lam": np.array([3.0, 4.0]),
    "semi_beta": np.array([600.0, 400.0]),
    "full_gamma": np.array([0.65, 0.45]),
    "price_elasticity": -1.2,
    "distribution_elasticity": 1.1,
}
display(pd.Series(truth_config, name="Known data-generating settings"))

### Independent data-generating response

**Notebook policy:** NumPy computes normalized causal adstock and both known responses without using the fitted terms graph. This provides an independent check of model wiring.


In [ ]:
def numpy_response(plan, form):
    """Return independent known-truth expected sales, in original units."""
    x = plan["spend"].transpose("date", "channel").values
    adstocked = np.empty_like(x)
    for c, alpha in enumerate(truth_config["alpha"]):
        weights = alpha ** np.arange(config["l_max"])
        weights /= weights.sum()
        adstocked[:, c] = np.convolve(x[:, c], weights)[: len(x)]
    shape = np.tanh(adstocked * truth_config["lam"] / 2)
    market = (plan["price"].values / price_ref) ** truth_config["price_elasticity"]
    market *= (plan["distribution"].values / distribution_ref) ** truth_config[
        "distribution_elasticity"
    ]
    if form == "semi":
        response = truth_config["baseline"] + (shape * truth_config["semi_beta"]).sum(
            axis=1
        )
    else:
        response = truth_config["baseline"] * np.exp(
            (np.log1p(shape) * truth_config["full_gamma"]).sum(axis=1)
        )
    return xr.DataArray(response * market, dims="date", coords={"date": plan["date"]})

Create a different noisy sales series for each response form. Divide sales by the fixed target scale, then separate training data from future inputs.


In [ ]:
truth = {form: numpy_response(inputs, form) for form in ("semi", "full")}
datasets = {
    form: inputs.assign(
        sales=(truth[form] + rng.normal(0, truth_config["noise_sigma"], n_total))
        / target_scale
    )
    for form in truth
}
train = {form: ds.isel(date=slice(0, n_train)) for form, ds in datasets.items()}
future = inputs.isel(date=slice(n_train, None))
historical_inputs = inputs.isel(date=slice(0, n_train))
display(train["semi"])

Plot the common inputs and both generated sales series before fitting.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True, layout="constrained")
for channel in channels:
    weekly_spend = (inputs["spend"] * channel_scale).sel(channel=channel)
    axes[0].plot(dates, weekly_spend.values, label=channel)
axes[0].legend(title="channel")
axes[0].set(
    ylabel="Weekly spend",
    title="Common inputs; independent variation aids identification",
)
axes[1].plot(dates, price / price_ref, label="Price / reference")
axes[1].plot(dates, distribution / distribution_ref, label="Distribution / reference")
axes[1].set(ylabel="Reference ratio")
axes[1].legend()
for form in truth:
    axes[2].plot(dates, datasets[form]["sales"] * target_scale, label=form, alpha=0.8)
axes[2].axvline(dates[n_train], color="black", linestyle=":")
axes[2].set(ylabel="Observed sales", xlabel="Week")
axes[2].legend()
plt.show()

<a id="models"></a>
## 3. Models

**Terms API:** compose labeled `Data`, transformation, parameter, and likelihood terms into a `GAM`. Both examples use a positive baseline and `Normal(mu, sigma)` observation noise. The mean is positive, but this likelihood can generate negative observations. These are not log-link likelihoods.

### Semi-multiplicative response

Media adds to the baseline; price and distribution multiply that sum:

\[
s_c(x)=\tanh\{\lambda_c\operatorname{adstock}_{\alpha_c}(x)/2\},\quad
A=b+\sum_c\beta_c s_c(x),\quad
M=(p/p_0)^{\gamma_p}(d/d_0)^{\gamma_d},\quad \mu=A M.
\]

The positive baseline and amplitudes keep the additive block positive.


Choose positive baseline and media-amplitude priors, channel-specific adstock and saturation priors, and market elasticity priors.


In [ ]:
semi_prior_config = {
    "baseline": Prior("Gamma", mu=1.0, sigma=0.4),
    "alpha": Prior("Beta", alpha=2, beta=5, dims="channel"),
    "lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
    "beta": Prior("HalfNormal", sigma=0.8, dims="channel"),
    "price": Prior("Normal", mu=0, sigma=1),
    "distribution": Prior("HalfNormal", sigma=1),
    "noise": Prior("HalfNormal", sigma=0.15),
}
display(
    pd.Series({k: repr(v) for k, v in semi_prior_config.items()}, name="Semi priors")
)

Compose adstock and saturation into a named media term. The next cell applies the market multiplier to each channel and the baseline before summing them into sales units.


In [ ]:
semi_spend = Data("spend")
semi_adstock = GeometricAdstock(
    l_max=config["l_max"], priors={"alpha": semi_prior_config["alpha"]}
)
semi_saturation = LogisticSaturation(
    priors={"lam": semi_prior_config["lam"], "beta": semi_prior_config["beta"]}
)
semi_media = (semi_spend >> semi_adstock >> semi_saturation).named("media_at_reference")
semi_baseline = Parameter("baseline", prior=semi_prior_config["baseline"])

Build the market multiplier. Name each conditional media effect and the baseline in sales units, sum them to `sales_units`, then divide by `target_scale` for the scaled likelihood mean.


In [ ]:
semi_price_gamma = Parameter("price_elasticity", prior=semi_prior_config["price"])
semi_distribution_gamma = Parameter(
    "distribution_elasticity", prior=semi_prior_config["distribution"]
)
semi_price_log = (
    semi_price_gamma * Transform(Data("price") * (1 / price_ref), ptx.math.log)
).named("price_log_factor")
semi_distribution_log = (
    semi_distribution_gamma
    * Transform(Data("distribution") * (1 / distribution_ref), ptx.math.log)
).named("distribution_log_factor")
semi_market = Transform(semi_price_log + semi_distribution_log, ptx.math.exp).named(
    "market_multiplier"
)
semi_media_units = (semi_media * semi_market * target_scale).named("media_sales_units")
semi_baseline_units = (semi_baseline * semi_market * target_scale).named(
    "baseline_sales_units"
)
semi_sales_units = (semi_baseline_units + semi_media_units.sum("channel")).named(
    "sales_units"
)
semi_response = (semi_sales_units * (1 / target_scale)).named("expected_sales")
semi_equation = Equation(
    name="y",
    observed="sales",
    mu=semi_response,
    likelihood=Prior("Normal", sigma=semi_prior_config["noise"]),
)
semi_gam = GAM(semi_equation)
semi_gam.build_model(train["semi"]);

### Fully multiplicative response

Use the same inference machinery but put each media term in a product with the market drivers:

\[
\ell_c=\gamma_c\log(1+s_c(x)),\quad
\mu=b\exp\left[\sum_c\ell_c+\gamma_p\log(p/p_0)+\gamma_d\log(d/d_0)\right].
\]

**Notebook recipe:** fix the existing logistic saturation's `beta=1.0` to expose a unit shape, then estimate `media_gamma` **outside** `log1p`. This is an explicit construction for #3088, not a generic shape-extraction API. Media exponents are dimensionless, rather than sales amplitudes. At zero media and reference market inputs, expected sales equal the fitted baseline.


Keep the same prior structure while replacing the sales-scale media amplitude with a dimensionless media exponent.


In [ ]:
full_prior_config = {
    "baseline": Prior("Gamma", mu=1.0, sigma=0.4),
    "alpha": Prior("Beta", alpha=2, beta=5, dims="channel"),
    "lam": Prior("Gamma", alpha=3, beta=1, dims="channel"),
    "gamma": Prior("HalfNormal", sigma=0.8, dims="channel"),
    "price": Prior("Normal", mu=0, sigma=1),
    "distribution": Prior("HalfNormal", sigma=1),
    "noise": Prior("HalfNormal", sigma=0.15),
}
display(
    pd.Series({k: repr(v) for k, v in full_prior_config.items()}, name="Full priors")
)

Build a unit saturation shape. Apply `log1p`, then put each channel exponent outside it before combining channel factors.


In [ ]:
full_spend = Data("spend")
full_adstock = GeometricAdstock(
    l_max=config["l_max"], priors={"alpha": full_prior_config["alpha"]}
)
full_saturation = LogisticSaturation(
    priors={"lam": full_prior_config["lam"], "beta": 1.0}
)
full_shape = (full_spend >> full_adstock >> full_saturation).named("unit_shape")
full_media_gamma = Parameter("media_gamma", prior=full_prior_config["gamma"])
full_media_log = (full_media_gamma * Transform(full_shape, ptx.math.log1p)).named(
    "media_log_factor"
)
full_media_factor = Transform(full_media_log.sum("channel"), ptx.math.exp).named(
    "media_multiplier"
)
full_baseline = Parameter("baseline", prior=full_prior_config["baseline"])

Add price and distribution to the product. Name total expected sales in original units, then divide by `target_scale` for the scaled likelihood mean. A single-channel sales contribution still needs a counterfactual definition.


In [ ]:
full_price_gamma = Parameter("price_elasticity", prior=full_prior_config["price"])
full_distribution_gamma = Parameter(
    "distribution_elasticity", prior=full_prior_config["distribution"]
)
full_price_log = (
    full_price_gamma * Transform(Data("price") * (1 / price_ref), ptx.math.log)
).named("price_log_factor")
full_distribution_log = (
    full_distribution_gamma
    * Transform(Data("distribution") * (1 / distribution_ref), ptx.math.log)
).named("distribution_log_factor")
full_market = Transform(full_price_log + full_distribution_log, ptx.math.exp).named(
    "market_multiplier"
)
full_sales_units = (
    full_baseline * full_media_factor * full_market * target_scale
).named("sales_units")
full_response = (full_sales_units * (1 / target_scale)).named("expected_sales")
full_equation = Equation(
    name="y",
    observed="sales",
    mu=full_response,
    likelihood=Prior("Normal", sigma=full_prior_config["noise"]),
)
full_gam = GAM(full_equation)
full_gam.build_model(train["full"]);

Inspect free parameter names and dimensions for both models; this catches unintended pooling.


In [ ]:
models = {"semi": semi_gam, "full": full_gam}
responses = {"semi": semi_response, "full": full_response}
for form, gam in models.items():
    print(form.upper(), "free parameters and dimensions")
    display(
        pd.Series(
            {rv.name: tuple(getattr(rv, "dims", ())) for rv in gam.model.free_RVs}
        )
    )

### Check both complete graphs against NumPy

Substitute known parameters before fitting. Compare each model's expected sales with its independent NumPy response. The fit below still estimates those parameters; it does not use their known values as observations.


Evaluate both graphs at known parameters and assert agreement with the independent data-generating responses.


In [ ]:
common_truth = {
    "baseline": truth_config["baseline"] / target_scale,
    "adstock_alpha": ptx.as_xtensor(truth_config["alpha"], dims=("channel",)),
    "saturation_lam": ptx.as_xtensor(truth_config["lam"], dims=("channel",)),
    "price_elasticity": truth_config["price_elasticity"],
    "distribution_elasticity": truth_config["distribution_elasticity"],
}
for form, gam in models.items():
    values = dict(common_truth)
    key = "saturation_beta" if form == "semi" else "media_gamma"
    amplitude = (
        truth_config["semi_beta"] / target_scale
        if form == "semi"
        else truth_config["full_gamma"]
    )
    values[key] = ptx.as_xtensor(amplitude, dims=("channel",))
    fixed = pm.do(gam.model, values)
    calculated = pm.draw(fixed["expected_sales"], random_seed=1) * target_scale
    np.testing.assert_allclose(
        calculated, truth[form].isel(date=slice(0, n_train)), rtol=1e-10
    )
print("Both complete response graphs match independent NumPy truth.")

<a id="fitting"></a>
## 4. Fitting and diagnostics

**Terms API:** draw from the prior, fit each `GAM`, and expose named posterior variables. **Review check:** inspect divergences, R-hat, and effective sample size before interpreting contributions or curves. The held-out weeks are absent from both fits.


Sample prior predictive sales and compare the model-implied sales ranges.


In [ ]:
priors = {
    form: gam.sample_prior_predictive(
        train[form], draws=200, var_names=["y"], random_seed=101
    )
    for form, gam in models.items()
}
prior_ranges = {}
for form, result in priors.items():
    sales = result["prior_predictive"]["y"].values.ravel() * target_scale
    prior_ranges[form] = dict(
        zip(["3%", "median", "97%"], np.quantile(sales, [0.03, 0.5, 0.97]), strict=True)
    )
display(pd.DataFrame(prior_ranges).T.rename_axis("Prior sales"))

Fit the semi-multiplicative model.


In [ ]:
semi_idata = semi_gam.fit(train["semi"], **sampler_config, random_seed=201)

Fit the fully multiplicative model with the same sampler settings.


In [ ]:
full_idata = full_gam.fit(train["full"], **sampler_config, random_seed=202)

Summarize divergences, maximum R-hat, and minimum bulk effective sample size for each fit.


In [ ]:
idatas = {"semi": semi_idata, "full": full_idata}
parameter_names = {
    "semi": [
        "baseline",
        "adstock_alpha",
        "saturation_lam",
        "saturation_beta",
        "price_elasticity",
        "distribution_elasticity",
        "y_sigma",
    ],
    "full": [
        "baseline",
        "adstock_alpha",
        "saturation_lam",
        "media_gamma",
        "price_elasticity",
        "distribution_elasticity",
        "y_sigma",
    ],
}
diagnostic_rows = []
for form, idata in idatas.items():
    posterior = idata["posterior"].to_dataset()[parameter_names[form]]
    diagnostic_rows.append(
        {
            "model": form,
            "divergences": int(idata["sample_stats"]["diverging"].sum()),
            "max_rhat": float(az.rhat(posterior).to_array().max()),
            "min_bulk_ess": float(az.ess(posterior, method="bulk").to_array().min()),
        }
    )
diagnostics = pd.DataFrame(diagnostic_rows).set_index("model")
display(diagnostics.round(3))
if (diagnostics["divergences"] > 0).any() or (diagnostics["max_rhat"] >= 1.01).any():
    print(
        "Inspect sampling: increase tuning/draws or adjust priors before interpreting the results."
    )

Inspect individual parameter summaries; aggregate diagnostics can hide a weakly sampled parameter.


In [ ]:
for form, idata in idatas.items():
    print(form.upper(), "posterior parameter summary (model units)")
    display(az.summary(idata, var_names=parameter_names[form], round_to=3))

<a id="avm"></a>
## 5. AVM and held-out forecast

**Terms API:** sample posterior predictions on labeled inputs. Training prediction starts at the simulation's zero-history boundary; the contiguous forecast includes the required training adstock history. **Notebook policy:** convert to sales units, plot actual versus model, and compute train/holdout errors.


Predict the training window and held-out future separately. For the forecast, prepend the needed adstock history.


In [ ]:
prediction_names = {
    "semi": [
        "y",
        "expected_sales",
        "sales_units",
        "baseline_sales_units",
        "media_sales_units",
        "baseline",
        "media_at_reference",
        "market_multiplier",
        "price_log_factor",
        "distribution_log_factor",
    ],
    "full": [
        "y",
        "expected_sales",
        "sales_units",
        "baseline",
        "unit_shape",
        "media_log_factor",
        "market_multiplier",
        "price_log_factor",
        "distribution_log_factor",
    ],
}
fitted, forecasts = {}, {}
for form, gam in models.items():
    fitted[form] = gam.sample_posterior_predictive(
        historical_inputs,
        var_names=prediction_names[form],
        include_last_observations=False,
        random_seed=301,
        progressbar=False,
    )
    forecasts[form] = gam.sample_posterior_predictive(
        future,
        var_names=["y", "expected_sales"],
        include_last_observations=True,
        random_seed=302,
        progressbar=False,
    )

Plot actual sales, posterior expected sales, predictive intervals, and known expected sales. Report train and holdout RMSE/MAE in original units.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, layout="constrained")
metrics = []
for ax, form in zip(axes, models, strict=True):
    prediction = (
        xr.concat([fitted[form][["y", "expected_sales"]], forecasts[form]], dim="date")
        * target_scale
    )
    interval = prediction["y"].quantile([0.03, 0.97], dim=("chain", "draw"))
    mean = prediction["expected_sales"].mean(("chain", "draw"))
    actual = datasets[form]["sales"] * target_scale
    ax.fill_between(
        dates,
        interval.sel(quantile=0.03),
        interval.sel(quantile=0.97),
        alpha=0.2,
        label="94% predictive interval",
    )
    ax.plot(dates, actual, color="0.35", lw=1, label="Actual")
    ax.plot(
        dates, truth[form], color="green", linestyle=":", label="Known expected sales"
    )
    ax.plot(dates, mean, color="C1", label="Posterior mean expected sales")
    ax.axvspan(dates[n_train], dates[-1], color="0.5", alpha=0.12, label="Held out")
    ax.set(title=f"{form.upper()}: actual versus model", ylabel="Sales units")
    ax.legend(ncol=2, fontsize=8)
    for split, window in {
        "training": slice(0, n_train),
        "holdout": slice(n_train, None),
    }.items():
        error = (mean - actual).isel(date=window)
        metrics.append(
            dict(
                model=form,
                split=split,
                RMSE=float(np.sqrt((error**2).mean())),
                MAE=float(abs(error).mean()),
            )
        )
axes[-1].set_xlabel("Week")
plt.show()
display(pd.DataFrame(metrics).set_index(["model", "split"]).round(2))

<a id="contributions"></a>
## 6. Contributions

Terms can expose named expressions and evaluate the fitted response under changed inputs. They can also compose original-unit sales and media-effect expressions in the reporting graph below. The analyst must choose what a *contribution* means, which inputs count as neutral, and how to allocate interactions. This section compares conditional components, leave-one-out effects, and Shapley allocations. These answer different questions.

### Conditional and leave-one-out effects

For the semi model, conditional media effects are \(a_cM\) and the no-media baseline is \(bM\). They add to \(\mu\), but price and distribution are already embedded in them. For the fully multiplicative model, removing channel \(c\) gives \(\mu[1-\exp(-\ell_c)]\). Those channel effects overlap. Market removal means restoring the declared references, never zero price or distribution.


Read the semi model’s named sales-unit baseline and media effects directly from posterior prediction and check their reconciliation. For the full model, calculate channel removal effects, which need not sum to total lift.


In [ ]:
semi_conditional_media = fitted["semi"]["media_sales_units"]
semi_conditional_baseline = fitted["semi"]["baseline_sales_units"]
for form in models:
    np.testing.assert_allclose(
        fitted[form]["sales_units"],
        fitted[form]["expected_sales"] * target_scale,
        rtol=1e-10,
    )
semi_conditional = xr.concat(
    [
        semi_conditional_baseline.expand_dims(component=["baseline at actual market"]),
        semi_conditional_media.rename(channel="component"),
    ],
    dim="component",
)
np.testing.assert_allclose(
    semi_conditional.sum("component"),
    fitted["semi"]["expected_sales"] * target_scale,
    rtol=1e-10,
)
full_removal_formula = (
    fitted["full"]["expected_sales"]
    * (1 - np.exp(-fitted["full"]["media_log_factor"]))
    * target_scale
)
display(
    semi_conditional.sum("date")
    .mean(("chain", "draw"))
    .to_series()
    .rename("Semi conditional sales")
)
display(
    full_removal_formula.sum("date")
    .mean(("chain", "draw"))
    .to_series()
    .rename("Full channel removal sales (overlapping)")
)

Plot the semi conditional components against expected sales to inspect draw-wise reconciliation visually.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5), layout="constrained")
conditional_table = (
    semi_conditional.mean(("chain", "draw")).transpose("date", "component").to_pandas()
)
ax.stackplot(
    conditional_table.index,
    conditional_table.T,
    labels=conditional_table.columns,
    alpha=0.8,
)
ax.plot(
    historical_inputs["date"],
    fitted["semi"]["expected_sales"].mean(("chain", "draw")) * target_scale,
    color="black",
    lw=1,
    label="Expected sales",
)
ax.set(
    title="Semi: reconciled media effects conditional on actual price and distribution",
    ylabel="Sales",
)
ax.legend(ncol=2, fontsize=8)
plt.show()

### Evaluate actual and neutral scenarios

**Terms API:** `Problem.evaluate` scores a labeled input plan on the same joint posterior draws. Its `terms` mapping can return expressions in sales units. **Notebook policy:** define the conversion and contribution expressions explicitly; this branch does not provide a generic `GAM` contribution or Shapley report. The objective only satisfies the `Problem` constructor; no optimization occurs until `.solve()`.

Players are TV, search, price, and distribution. Removing media sets its spend to zero over the **whole training history**; removing a market driver restores its fixed reference. All scenarios share zero prehistory. Thus this measures a whole-history intervention, not a change confined to an arbitrary sub-window.


Compile one reporting `Problem` per fitted response. Its term mapping exposes scaled response, expected sales in original units, and model-specific media effects in original units. For the semi model those are conditional components; for the full model they are overlapping single-channel removal effects. Its objective is only a constructor input for scenario evaluation.


In [ ]:
def expected_total(evaluate, u):
    """Return posterior mean total sales in original units."""
    return (evaluate(u)["response"] * target_scale).sum("date").mean("sample")


scenario_bounds = {
    "spend": (0.0, 2.0),
    "price": (0.7 * price_ref, 1.3 * price_ref),
    "distribution": (0.5, 1.0),
}
reporting_terms = {
    "semi": {
        "response": semi_response,  # model units: thousands of sales
        "sales_units": semi_sales_units,
        "media_units": semi_media_units,
        "baseline_units": semi_baseline_units,
    },
    "full": {
        "response": full_response,
        "sales_units": full_sales_units,
        # This sales-unit removal term is defined for the reporting graph.
        "media_units": (
            full_sales_units * (1 - Transform(-full_media_log, ptx.math.exp))
        ).named("media_removal_sales_units"),
    },
}
reporting_problems = {
    form: Problem(
        model=gam,
        terms=reporting_terms[form],
        dates=pd.DatetimeIndex(historical_inputs["date"].values),
        objective=expected_total,
        bounds=scenario_bounds,
    )
    for form, gam in models.items()
}

Define the four players and one neutralization rule per input. These choices determine every counterfactual below.


In [ ]:
players = [*channels, "price", "distribution"]
all_mask = (1 << len(players)) - 1


def coalition_plan(mask, actual):
    """Keep enabled players at actual inputs; put others at neutral inputs."""
    plan = actual.copy(deep=True)
    enabled = xr.DataArray(
        [bool(mask & (1 << i)) for i in range(len(channels))],
        dims="channel",
        coords={"channel": channels},
    )
    plan["spend"] = plan["spend"] * enabled
    if not mask & (1 << len(channels)):
        plan["price"] = xr.full_like(plan["price"], price_ref)
    if not mask & (1 << (len(channels) + 1)):
        plan["distribution"] = xr.full_like(plan["distribution"], distribution_ref)
    return plan

Evaluate all 16 coalitions per model. Check that the all-actual coalition matches posterior prediction on the same draws.


In [ ]:
coalition_values = {}
for form, problem in reporting_problems.items():
    coalition_values[form] = {
        mask: problem.evaluate(coalition_plan(mask, historical_inputs))["sales_units"]
        for mask in range(all_mask + 1)
    }
    native_prediction = (
        (fitted[form]["expected_sales"] * target_scale)
        .stack(sample=("chain", "draw"))
        .transpose("sample", "date")
    )
    np.testing.assert_allclose(
        coalition_values[form][all_mask], native_prediction, rtol=1e-9, atol=1e-7
    )
    print(
        form,
        "all",
        len(coalition_values[form]),
        "coalitions evaluated on identical joint posterior draws",
    )

Compare each all-actual response with the response after removing one player. Their sum can exceed joint lift because interactions overlap.


In [ ]:
removal_effects = {}
removal_rows = []
for form, values in coalition_values.items():
    removal = xr.concat(
        [values[all_mask] - values[all_mask ^ (1 << i)] for i in range(len(players))],
        dim=xr.IndexVariable("driver", players),
    )
    removal_effects[form] = removal
    joint_lift = values[all_mask] - values[0]
    excess = removal.sum("driver") - joint_lift
    removal_rows.append(
        dict(
            model=form,
            joint_lift=float(joint_lift.sum("date").mean("sample")),
            sum_removal_effects=float(removal.sum(("driver", "date")).mean("sample")),
            overlap=float(excess.sum("date").mean("sample")),
        )
    )
display(pd.DataFrame(removal_rows).set_index("model").round(2))
print("Overlap is allowed. Removal effects are questions, not a conserving allocation.")

Check both hand-derived media removal formulas against full-graph counterfactual evaluation.


In [ ]:
for form, formula in {
    "semi": semi_conditional_media,
    "full": full_removal_formula,
}.items():
    calculated = formula.stack(sample=("chain", "draw")).transpose(
        "sample", "date", "channel"
    )
    evaluated = (
        removal_effects[form]
        .sel(driver=channels)
        .rename(driver="channel")
        .transpose("sample", "date", "channel")
    )
    np.testing.assert_allclose(evaluated, calculated, rtol=1e-9, atol=1e-7)
    graph_units = reporting_problems[form].evaluate(historical_inputs)
    np.testing.assert_allclose(
        graph_units["media_units"].transpose("sample", "date", "channel"),
        calculated,
        rtol=1e-9,
        atol=1e-7,
    )
    np.testing.assert_allclose(
        graph_units["sales_units"],
        coalition_values[form][all_mask],
        rtol=1e-9,
        atol=1e-7,
    )
    if form == "semi":
        np.testing.assert_allclose(
            graph_units["baseline_units"] + graph_units["media_units"].sum("channel"),
            graph_units["sales_units"],
            rtol=1e-9,
            atol=1e-7,
        )
print("Both model-specific media formulas agree with full-graph channel removal.")

### Allocate interactions with exact Shapley values

**Notebook policy:** enumerate all \(2^4=16\) coalitions for four players, then allocate the fitted mean on every posterior draw and date:

\[
\phi_i=\sum_{S\subseteq N\setminus\{i\}}
\frac{|S|!(n-|S|-1)!}{n!}\,[v(S\cup\{i\})-v(S)].
\]

The neutral baseline is \(v(\varnothing)\). It plus all allocations reconstructs expected sales, not noisy observed sales. The answer depends on player grouping and references; price or distribution allocations can be negative. Exact enumeration grows exponentially, so larger games need a separately validated approximation. This is attribution under the model, not causal identification.


Allocate each coalition game across its four players, separately for every posterior draw and date.


In [ ]:
def exact_shapley(values, names):
    """Allocate a cached coalition game separately for every posterior draw and date."""
    n = len(names)
    allocated = []
    for i in range(n):
        effect = xr.zeros_like(values[0])
        for mask in range(1 << n):
            if mask & (1 << i):
                continue
            size = mask.bit_count()
            weight = factorial(size) * factorial(n - size - 1) / factorial(n)
            effect = effect + weight * (values[mask | (1 << i)] - values[mask])
        allocated.append(effect)
    return xr.concat(allocated, dim=xr.IndexVariable("driver", names))


shapley = {
    form: exact_shapley(values, players) for form, values in coalition_values.items()
}

Assert that neutral sales plus Shapley allocations recover the all-actual response. Also test the allocation rule on a two-player toy game.


In [ ]:
for form, allocated in shapley.items():
    values = coalition_values[form]
    reconstructed = values[0] + allocated.sum("driver")
    np.testing.assert_allclose(reconstructed, values[all_mask], rtol=1e-10, atol=1e-7)
    print(
        form,
        "maximum draw/date reconciliation error:",
        float(abs(reconstructed - values[all_mask]).max()),
    )

# A tiny game also checks the allocation rule independently of either fitted model.
toy = {
    mask: xr.DataArray((100 + 20 * bool(mask & 1)) * (1.5 if mask & 2 else 1.0))
    for mask in range(4)
}
np.testing.assert_allclose(exact_shapley(toy, ["media", "market"]), [25.0, 55.0])

### Compare with known-policy allocation

Apply the same whole-history coalition policy to the independent synthetic truth. This checks recovery under a matched policy; it does not make Shapley the uniquely correct contribution definition.


Compare posterior Shapley totals, known-truth allocations, and overlapping removal totals in one table.


In [ ]:
truth_coalitions = {
    form: {
        mask: numpy_response(coalition_plan(mask, historical_inputs), form)
        for mask in range(all_mask + 1)
    }
    for form in models
}
truth_shapley = {
    form: exact_shapley(values, players) for form, values in truth_coalitions.items()
}
allocation_tables = {}
for form, allocated in shapley.items():
    totals = allocated.sum("date")
    interval = totals.quantile([0.03, 0.97], dim="sample")
    allocation_tables[form] = pd.DataFrame(
        {
            "Shapley mean": totals.mean("sample").to_series(),
            "3%": interval.sel(quantile=0.03).to_series(),
            "97%": interval.sel(quantile=0.97).to_series(),
            "Known-policy allocation": truth_shapley[form].sum("date").to_series(),
            "Removal mean (overlapping)": removal_effects[form]
            .sum("date")
            .mean("sample")
            .to_series(),
        }
    )
    print(form.upper())
    display(allocation_tables[form].round(1))

Plot total Shapley allocations with posterior intervals and known-policy values.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for ax, (form, table) in zip(axes, allocation_tables.items(), strict=True):
    y = np.arange(len(table))
    ax.hlines(y, table["3%"], table["97%"], color="C0", lw=4, alpha=0.5)
    ax.scatter(table["Shapley mean"], y, label="Posterior mean")
    ax.scatter(
        table["Known-policy allocation"],
        y,
        marker="x",
        color="black",
        label="Known policy allocation",
    )
    ax.set(
        yticks=y,
        yticklabels=table.index,
        xlabel="Sales over training period",
        title=f"{form.upper()}: Shapley, 94% interval",
    )
    ax.axvline(0, color="0.4", lw=1)
    ax.legend(fontsize=8)
plt.show()

Plot signed allocations through time. A negative market allocation is possible under this reference policy.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, layout="constrained")
for ax, form in zip(axes, shapley, strict=True):
    for player in players:
        ax.plot(
            historical_inputs["date"],
            shapley[form].sel(driver=player).mean("sample"),
            label=player,
        )
    ax.axhline(0, color="0.5", lw=1)
    ax.set(
        title=f"{form.upper()}: signed Shapley sales allocations", ylabel="Sales / week"
    )
    ax.legend(ncol=4)
axes[-1].set_xlabel("Week")
plt.show()

**Review note:** the optimizer below evaluates the response directly. Shapley is a reporting choice and does not enter its objective.


<a id="curves"></a>
## 7. Curves and returns

### Component curves

These curves use **adstocked input**, not an eight-week spend plan. The semi model's component has a sales amplitude; the fully multiplicative component is a dimensionless factor. Multiplying that factor by `target_scale` would not create a sales contribution.


Plot each component on adstocked spend. Read the y-axis carefully: semi uses sales units; full uses a unitless factor.


In [ ]:
shape_grid = xr.DataArray(
    np.linspace(0, 1.5, 60), dims="adstocked_spend", name="adstocked_spend"
)
component_curves = {}
for form, idata in idatas.items():
    post = idata["posterior"].to_dataset()
    unit_shape = np.tanh(post["saturation_lam"] * shape_grid / 2)
    component_curves[form] = (
        post["saturation_beta"] * unit_shape * target_scale
        if form == "semi"
        else np.exp(post["media_gamma"] * np.log1p(unit_shape))
    )
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
for ax, form in zip(axes, models, strict=True):
    for channel in channels:
        curve = component_curves[form].sel(channel=channel)
        ax.plot(
            shape_grid * channel_scale.sel(channel=channel).item(),
            curve.mean(("chain", "draw")),
            label=channel,
        )
    ax.set(
        title=f"{form.upper()}: component only",
        xlabel="Adstocked spend (currency)",
        ylabel="Reference media effect (sales)"
        if form == "semi"
        else "Media factor (dimensionless)",
    )
    ax.legend()
plt.show()

### Scenario response curves in sales units

**Notebook policy:** each point is eight weeks of constant channel spend plus `l_max - 1` zero-spend tail weeks, with zero prehistory. Hold the other channel at its historical mean during the decision weeks and zero in the tail; hold distribution at its reference. Set price to 90%, 100%, or 110% of reference for the full horizon.

Subtract a *paired* zero-spend response for the swept channel, keeping all other inputs fixed. Sum over the decision weeks and tail, then divide by eight. The result is average incremental sales per decision week including carryover. It is a conditional removal curve, not Shapley; market and other-channel context affect it.


Create a separate evaluator over the eight-week decision horizon and five-week carryover tail.


In [ ]:
curve_weeks = config["curve_weeks"]
tail_weeks = config["l_max"] - 1
curve_dates = pd.date_range(
    dates[n_train], periods=curve_weeks + tail_weeks, freq="W-MON"
)
curve_problems = {
    form: Problem(
        model=gam,
        terms={"response": responses[form]},
        dates=curve_dates,
        objective=expected_total,
        bounds=scenario_bounds,
    )
    for form, gam in models.items()
}
weekly_spend = historical_inputs["spend"].mean("date")
price_contexts = [0.9, 1.0, 1.1]

Build a constant weekly plan with explicit zero-spend tail, fixed distribution, and chosen price context.


In [ ]:
def constant_weekly_plan(template, price_ratio=1.0):
    """Create the declared eight-week plan and its zero-spend tail."""
    plan = template.copy(deep=True)
    active = xr.DataArray(
        np.arange(len(curve_dates)) < curve_weeks,
        dims="date",
        coords={"date": curve_dates},
    )
    plan["spend"] = weekly_spend.broadcast_like(plan["spend"]) * active
    plan["price"] = xr.full_like(plan["price"], price_ref * price_ratio)
    plan["distribution"] = xr.full_like(plan["distribution"], distribution_ref)
    return plan

Sweep each channel over its observed spend range. At every price context, compare the same posterior draws for a plan and its zero-channel-spend counterfactual.


In [ ]:
response_curves, known_response_curves = {}, {}
for form, problem in curve_problems.items():
    channel_curves, known_channel_curves = [], []
    for channel in channels:
        context_curves, known_context_curves = [], []
        grid = np.linspace(
            0,
            float(historical_inputs["spend"].sel(channel=channel).max()),
            config["curve_points"],
        )
        for ratio in price_contexts:
            base_plan = constant_weekly_plan(problem.decisions, ratio)
            base_plan["spend"].loc[{"channel": channel}] = 0.0
            zero = problem.evaluate(base_plan)["response"] * target_scale
            known_zero = numpy_response(base_plan, form)
            draws, known = [], []
            for spend in grid:
                plan = base_plan.copy(deep=True)
                plan["spend"].loc[{"channel": channel}] = np.r_[
                    np.full(curve_weeks, spend), np.zeros(tail_weeks)
                ]
                draws.append(
                    (problem.evaluate(plan)["response"] * target_scale - zero).sum(
                        "date"
                    )
                    / curve_weeks
                )
                known.append(
                    float(
                        (numpy_response(plan, form) - known_zero).sum("date")
                        / curve_weeks
                    )
                )
            spend_axis = xr.IndexVariable(
                "weekly_spend", grid * channel_scale.sel(channel=channel).item()
            )
            context_curves.append(xr.concat(draws, dim=spend_axis))
            known_context_curves.append(
                xr.DataArray(
                    known, dims="weekly_spend", coords={"weekly_spend": spend_axis}
                )
            )
        channel_curves.append(
            xr.concat(
                context_curves, dim=xr.IndexVariable("price_ratio", price_contexts)
            )
        )
        known_channel_curves.append(
            xr.concat(
                known_context_curves,
                dim=xr.IndexVariable("price_ratio", price_contexts),
            )
        )
    # Keep a mapping: channels have different currency grids and must not be outer-aligned into one grid.
    response_curves[form] = dict(zip(channels, channel_curves, strict=True))
    known_response_curves[form] = dict(zip(channels, known_channel_curves, strict=True))

Plot sales-scale response curves with posterior intervals and the independently generated known response.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
for row, form in enumerate(models):
    for col, channel in enumerate(channels):
        ax = axes[row, col]
        for color, ratio in enumerate(price_contexts):
            curve = response_curves[form][channel].sel(price_ratio=ratio)
            interval = curve.quantile([0.03, 0.97], dim="sample")
            ax.fill_between(
                curve["weekly_spend"],
                interval.sel(quantile=0.03),
                interval.sel(quantile=0.97),
                color=f"C{color}",
                alpha=0.12,
            )
            ax.plot(
                curve["weekly_spend"],
                curve.mean("sample"),
                color=f"C{color}",
                label=f"Price {ratio:.0%} reference",
            )
            ax.plot(
                curve["weekly_spend"],
                known_response_curves[form][channel].sel(price_ratio=ratio),
                color=f"C{color}",
                linestyle=":",
            )
        ax.set(
            title=f"{form.upper()} — {channel}",
            xlabel="Weekly channel spend (currency)",
            ylabel="Incremental sales / decision week",
        )
        ax.legend(fontsize=8)
fig.suptitle(
    "Conditional response curves: 94% posterior intervals; dotted = known response"
)
plt.show()

### Average and marginal return curves

**Notebook policy:** divide conditional incremental sales by weekly spend for average ROAS, and differentiate the curve with respect to weekly spend for marginal ROAS. Both are in sales units per currency unit. Zero-spend ROAS is undefined and excluded. Neither calculation uses Shapley allocations.


Derive average and finite-difference marginal ROAS from the response curves at reference price.


In [ ]:
return_curves = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 6), layout="constrained")
for row, form in enumerate(models):
    for channel in channels:
        curve = response_curves[form][channel].sel(price_ratio=1.0)
        positive = curve.sel(weekly_spend=curve["weekly_spend"] > 0)
        roas = positive / positive["weekly_spend"]
        marginal = curve.differentiate("weekly_spend")
        return_curves[form, channel] = {"roas": roas, "marginal": marginal}
        axes[row, 0].plot(roas["weekly_spend"], roas.mean("sample"), label=channel)
        axes[row, 1].plot(
            marginal["weekly_spend"], marginal.mean("sample"), label=channel
        )
    for col, metric in enumerate(["Average ROAS", "Marginal ROAS (finite difference)"]):
        axes[row, col].set(
            title=f"{form.upper()}: {metric}",
            xlabel="Weekly spend",
            ylabel="Sales units / currency unit",
        )
        axes[row, col].legend()
plt.show()

Check zero-spend response and the paired price-elasticity identity for each model and channel.


In [ ]:
for form in models:
    for channel in channels:
        curve = response_curves[form][channel]
        np.testing.assert_allclose(curve.isel(weekly_spend=0), 0.0, atol=1e-7)
        reference = curve.sel(price_ratio=1.0)
        # A direct elasticity identity checks scenario context and paired draws.
        gamma = idatas[form]["posterior"]["price_elasticity"].stack(
            sample=("chain", "draw")
        )
        np.testing.assert_allclose(
            curve.sel(price_ratio=0.9), reference * (0.9**gamma), rtol=1e-8, atol=1e-7
        )
print("Zero-spend and paired price-elasticity curve checks pass.")

<a id="optimization"></a>
## 8. Optimization

**Terms API:** reuse the fitted response with `Problem.evaluate` and `Problem.solve`. **Notebook policy:** maximize expected sales over eight decision weeks and a fixed zero-spend tail, with zero prehistory, a fixed media budget, fixed reference price/distribution, and historical spend bounds. The baseline is constant across these plans. This is a local constrained optimum, not a global-optimality or causal budget claim.


Declare the original-currency budget equality. Solve a plan for each model over decided weeks while the tail remains fixed.


In [ ]:
def fixed_budget(evaluate, u):
    """Return the original-currency budget residual for equality to zero."""
    spend = u["spend"] * evaluate.constant(channel_scale)
    return evaluate.parameter("budget") - spend.sum("date").sum("channel")


optimization_results = {}
if RUN_OPTIMIZATION:
    decision_dates = curve_dates[:curve_weeks]
    fixed_tail = constant_weekly_plan(curve_problems["semi"].decisions).isel(
        date=slice(curve_weeks, None)
    )
    budget = float((weekly_spend * channel_scale).sum()) * curve_weeks
    planning_bounds = {
        "spend": (0.0, historical_inputs["spend"].max("date")),
        "price": (price_ref, price_ref),
        "distribution": (distribution_ref, distribution_ref),
    }
    for form, gam in models.items():
        problem = Problem(
            model=gam,
            terms={"response": responses[form]},
            dates=decision_dates,
            objective=expected_total,
            bounds=planning_bounds,
            constraints=[{"type": "eq", "fun": fixed_budget}],
            parameters={"budget": budget},
            carryover=fixed_tail,
        )
        initial = constant_weekly_plan(curve_problems[form].decisions).isel(
            date=slice(0, curve_weeks)
        )
        result = problem.solve(x0=initial)
        optimization_results[form] = (problem, initial, result)
        print(form, result)

Inspect the public solver result, allocated currency, feasibility, and expected-sales change. Solver success alone does not prove feasibility.


In [ ]:
if RUN_OPTIMIZATION:
    planning_rows = []
    for form, (problem, initial, result) in optimization_results.items():
        initial_sales = float(
            (problem.evaluate(initial)["response"] * target_scale)
            .sum("date")
            .mean("sample")
        )
        # Inspect the public result fields rather than treating solver success as feasibility.
        print(form, "result fields:", list(vars(result)))
        display(
            (result.allocation["spend"] * channel_scale)
            .sum("date")
            .to_series()
            .rename("Allocated currency")
        )
        planning_rows.append(
            {
                "model": form,
                "initial_sales": initial_sales,
                "optimized_sales": result.objective,
                "feasible": result.feasible,
                "solver_success": result.scipy.success,
            }
        )
    display(pd.DataFrame(planning_rows).set_index("model").round(2))

<a id="review"></a>
## 9. Persistence and review takeaways

### Optional model save/load

**Terms API:** `GAM.save` / `GAM.load` store and reconstruct the fitted model recipe and posterior. **Notebook policy:** original-unit scales, player grouping, neutral references, and scenario rules also need to travel with the analysis. The optional bundle writes those settings separately. `RUN_SAVE_LOAD=False` by default, so this cell is an available check rather than evidence from the normal run.


If enabled, save each fitted model and a separate JSON file containing the analysis policy. Reload and compare its forecast with the original run.


In [ ]:
if RUN_SAVE_LOAD:
    import json

    output_dir = Path("local_terms_models")
    output_dir.mkdir(exist_ok=True)
    metadata = {
        "config": config,
        "channels": channels,
        "channel_scale": channel_scale.values.tolist(),
        "players": players,
        "attribution": "exact whole-history Shapley",
        "response": "conditional mean sales",
    }
    (output_dir / "analysis_settings.json").write_text(json.dumps(metadata, indent=2))
    for form, gam in models.items():
        gam.save(output_dir / f"{form}.zarr")
        loaded = GAM.load(output_dir / f"{form}.zarr")
        restored = loaded.sample_posterior_predictive(
            future,
            var_names=["expected_sales"],
            include_last_observations=True,
            progressbar=False,
        )
        np.testing.assert_allclose(
            restored["expected_sales"], forecasts[form]["expected_sales"], rtol=1e-9
        )
    print("Local model bundles written to", output_dir.resolve())

### PR review summary

| Area | Supplied by terms on this branch | Still defined or calculated here |
| --- | --- | --- |
| Model and units | `Data`, `Parameter`, `Transform`, `Equation`, `GAM`, priors and `.named()` deterministics; `Problem.terms` can expose additional outputs | Fixed scales and references; sales-unit terms composed explicitly; the full model’s unit-shape construction |
| Fit and prediction | Prior predictive, posterior fit, posterior prediction, adstock history | Diagnostics interpretation, AVM metrics and plots, conversion back to sales units |
| Take contributions | Named graph outputs and `Problem.evaluate` on joint posterior draws | Conditional components, whole-history removal, neutral baseline, player grouping, exact Shapley allocation, and reconciliation |
| Curves and returns | Evaluate the fitted response on new labeled plans | Spend grid, market context, carryover horizon, zero-spend comparison, ROAS and marginal ROAS |
| Planning | `Problem.evaluate` and `Problem.solve` | Objective, budget, bounds, fixed controls, tail policy, and interpretation of a local solution |
| Persistence | `GAM.save` / `GAM.load` | Scales, references, grouping, and attribution policy stored separately; optional check disabled by default |

**Questions for the PR review:** Should the experimental API standardize any contribution estimand, or leave conditional effects, removals, and interaction allocation to applications? If results are to be reusable, where should units, reference values, intervention windows, and player groups live? Which AVM, curve, and return summaries belong in a higher-level results layer? The checks here demonstrate what can be assembled; they do not make those policy choices for other models.


In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pymc,pymc_extras,pytensor,arviz,xarray